# 6.2 — Vectors in a DataFrame

**Chapter 6, section 6.1.5**, and the starting point for **Exercise 5**.

**The question this notebook answers:** a machine learning workflow has to carry a *vector* in a
DataFrame column. Which type does it use, and what does it still have after a NumPy operation has
touched it?

The answer is `Vectors` from `pyspark.ml.linalg`, and the boundary between it and NumPy is porous
in one direction only. NumPy arithmetic on a Spark vector **returns a NumPy array**, not a Spark
vector, and that one-way traffic is the whole subject here. The notebook ends on the trap the
chapter's note box describes: two classes named `Vectors`, in two namespaces, whose error message
names types that look identical.

Source: `Notebooks old/Spark-Example-07-RDD-Dataframe-Examples-Operations.ipynb`, cells 57–59,
rebuilt. That file's operator survey became `03.02 RDD and DataFrame Side by Side.ipynb` in
chapter 3, which did not keep these closing cells; they are chapter 6 material and get their own
notebook here.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
import numpy as np
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-6.2")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

rng = np.random.default_rng(777)      # fixed seed: every number below is reproducible

print("Spark", spark.version, "| NumPy", np.__version__)

Spark 4.2.0 | NumPy 2.5.2


---

## 1. The two-column convention

Inside a DataFrame the case for reaching for NumPy is narrower than it first appears: a Python
function applied to a column runs as a user-defined function, on the wrong side of the boundary
between the JVM and the Python interpreter, where Catalyst cannot see it. What is left is a
different question — how a *vector* is stored in a column at all.

In [2]:
from pyspark.ml.linalg import Vectors

size = 2
data = [(0, Vectors.dense(rng.random(size)),),
        (1, Vectors.dense(rng.random(size)),),
        (1, Vectors.dense(rng.random(size)),),
        (0, Vectors.dense(rng.random(size)),)]

df = spark.createDataFrame(data, ["label", "features"])
df.show(truncate=False)
df.printSchema()

+-----+-----------------------------------------+
|label|features                                 |
+-----+-----------------------------------------+
|0    |[0.6110939299469712,0.38281659045082816] |
|1    |[0.6000705254490022,0.9635578674390097]  |
|1    |[0.19616256242654817,0.33704142186654373]|
|0    |[0.5684749820146611,0.5395921034316342]  |
+-----+-----------------------------------------+

root
 |-- label: long (nullable = true)
 |-- features: vector (nullable = true)



One row per training example, and the `features` column holds **a single vector per row** rather
than one numeric column per feature. The schema is the thing to read: `features` is not an array
of doubles but a `VectorUDT`, a user-defined type Spark's machine learning library understands.

The layout looks redundant until the width of real feature vectors is considered. A model over ten
thousand features would otherwise need ten thousand columns, and every MLlib estimator is instead
specified against exactly this two-column convention — one label and one assembled vector.
`Vectors.dense` accepts a NumPy array directly, which is the interoperability the chapter promised.

In practice such vectors are not written by hand; they are assembled from raw columns by a
featurizer. `VectorAssembler` is the one to know, and it produces the identical layout.

In [3]:
from pyspark.ml.feature import VectorAssembler

raw = spark.createDataFrame(
    [(0, 1.0, 2.0), (1, 3.0, 4.0), (1, 5.0, 6.0), (0, 7.0, 8.0)],
    ["label", "x1", "x2"])

assembled = VectorAssembler(inputCols=["x1", "x2"], outputCol="features").transform(raw)
assembled.select("label", "features").show(truncate=False)
print(assembled.schema["features"].dataType)

+-----+---------+
|label|features |
+-----+---------+
|0    |[1.0,2.0]|
|1    |[3.0,4.0]|
|1    |[5.0,6.0]|
|0    |[7.0,8.0]|
+-----+---------+

VectorUDT()


---

## 2. The boundary, in the friendly direction

NumPy arrays travel into Spark easily. An RDD whose records are NumPy rows reduces with `np.add`
exactly as it did with a lambda in notebook 6.1 — a named function in place of the earlier
anonymous one.

In [4]:
# re-seeded here so this block reproduces on its own, exactly as the notes print it
rng = np.random.default_rng(777)

mat = rng.random((4, 2))
print(mat)

rdd = sc.parallelize(mat)
total = rdd.reduce(lambda x, y: np.add(x, y))
print("reduced     :", total)
print("column sums :", mat.sum(axis=0))
assert np.allclose(total, mat.sum(axis=0))

[[0.61109393 0.38281659]
 [0.60007053 0.96355787]
 [0.19616256 0.33704142]
 [0.56847498 0.5395921 ]]


reduced     : [1.975802   2.22300798]
column sums : [1.975802   2.22300798]


MLlib's own vector types participate in NumPy arithmetic too, converting on the way out.

In [5]:
a = Vectors.dense(np.round(rng.random(size), 2))
b = Vectors.dense(np.round(rng.random(size), 2))
print("a =", a, " type:", type(a).__name__)
print("b =", b, " type:", type(b).__name__)

s = np.add(a, b)
print("np.add(a, b) =", s, " type:", type(s).__name__)

a = [0.74,0.43]  type: DenseVector
b = [0.06,0.17]  type: DenseVector
np.add(a, b) = [0.8 0.6]  type: ndarray


That is the fact worth keeping. Two `DenseVector` objects go in; **a plain NumPy array comes
out**. Convenient as this is at a notebook prompt, the direction of the conversion is what
matters: NumPy operations produce NumPy values, so code that must keep a `features` column typed
as a Spark vector has to construct `Vectors` explicitly rather than trust a round trip.

In [6]:
back = Vectors.dense(s)               # the explicit reconstruction
print(type(s).__name__, "->", type(back).__name__)

# and the same thing said as an assertion that fails loudly if a release ever changes it
assert isinstance(np.add(a, b), np.ndarray)
assert not isinstance(np.add(a, b), type(a))

ndarray -> DenseVector


---

## 3. Two `Vectors` classes, one name

`pyspark.ml.linalg.Vectors` and `pyspark.mllib.linalg.Vectors` are **distinct classes with
distinct types**, and they are not interchangeable. This is the chapter's note box, and it is the
subject of Exercise 5.

In [7]:
from pyspark.ml.linalg import Vectors as ML_Vectors
from pyspark.mllib.linalg import Vectors as MLlib_Vectors

v_ml = ML_Vectors.dense([1.0, 2.0])
v_mllib = MLlib_Vectors.dense([1.0, 2.0])

print("ml    :", v_ml, type(v_ml))
print("mllib :", v_mllib, type(v_mllib))
print("\nthey print the same. But:")
print("  v_ml == v_mllib      ->", v_ml == v_mllib)
print("  same class           ->", type(v_ml) is type(v_mllib))
print("  same values in NumPy ->", (v_ml.toArray() == v_mllib.toArray()).all())

ml    : [1.0,2.0] <class 'pyspark.ml.linalg.DenseVector'>
mllib : [1.0,2.0] <class 'pyspark.mllib.linalg.DenseVector'>

they print the same. But:
  v_ml == v_mllib      -> False
  same class           -> False
  same values in NumPy -> True


They print identically. Worse, a DataFrame built from the wrong one **builds without complaint**
and `show()` looks exactly right. The failure is deferred to the estimator.

In [8]:
wrong = spark.createDataFrame(
    [(0.0, MLlib_Vectors.dense([1.0, 2.0])), (1.0, MLlib_Vectors.dense([3.0, 4.0]))],
    ["label", "features"])

wrong.show()                                   # looks correct
print(wrong.schema["features"].dataType)       # and even the schema says VectorUDT

+-----+---------+
|label| features|
+-----+---------+
|  0.0|[1.0,2.0]|
|  1.0|[3.0,4.0]|
+-----+---------+

VectorUDT()


In [9]:
from pyspark.ml.classification import LogisticRegression

try:
    LogisticRegression(maxIter=1).fit(wrong)
except Exception as e:
    print(type(e).__name__, ":", str(e).splitlines()[0])

IllegalArgumentException : requirement failed: Column features must be of type class org.apache.spark.ml.linalg.VectorUDT:struct<type:tinyint,size:int,indices:array<int>,values:array<double>> but was actually class org.apache.spark.mllib.linalg.VectorUDT:struct<type:tinyint,size:int,indices:array<int>,values:array<double>>.


Read that message carefully, because its unhelpfulness is the lesson: the two type names differ
only in `ml` against `mllib`, and their struct layouts, printed side by side in the same sentence,
are **character for character identical**. Nothing in the message says "you imported from the
wrong package".

The one-line fix is the import.

In [10]:
fixed = spark.createDataFrame(
    [(0.0, ML_Vectors.dense([1.0, 2.0])), (1.0, ML_Vectors.dense([3.0, 4.0]))],
    ["label", "features"])

model = LogisticRegression(maxIter=1).fit(fixed)
print("fit succeeded; coefficients:", model.coefficients)

fit succeeded; coefficients: [0.5,0.5]


---

## Conclusion

**The working rule.** Code that feeds a DataFrame-based estimator imports from
`pyspark.ml.linalg`. Code that constructs a distributed matrix — notebook 6.4 — imports from
`pyspark.mllib.linalg` and its `distributed` subpackage, because that is the only namespace in
which the distributed types exist. When an import error mentions a vector type that appears to be
the one in use, the first thing to check is which of the two namespaces each side of the call came
from.

**And the boundary.** A `features` column holds a Spark vector. NumPy will happily compute with
it and will hand back a NumPy array, so anything that has to stay a Spark vector must be rebuilt
as one on purpose.

In [11]:
spark.stop()